In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

File is mostly commented out because it is dependent on structure of private, health data

In [ ]:
'''
#Read in demographics, history, and ses variables
demographics = pd.read_csv('demographics.csv')
off = pd.read_csv('office_visits.csv')
ses = pd.read_csv('ses.csv')

#merge into one dataframe
demographics = demographics.merge(off, right_on = 'preg_id', left_on = 'preg_id', how='left')
demographics = demographics.merge(ses, right_on = 'preg_id', left_on = 'preg_id', how='left')
'''

In [3]:
'''
#### USER INPUTS MOTHER OUTCOMES ####

#at birth outcome (if we need to cut off diagnoses or not)
at_birth = False

#include SES and racial variables in predictions
ses = True

#Features used and diag cut off if at birth = False
cutoff = 12

##GESTATIONAL HYPERTENSTION##
diag_col = 'ga_ghtn'
diag = 'ghtn'
outcome = 'ghtn'

ghtn =  demographics[demographics['ghtn'] == 1]


##GDM##
diag_col = 'ga_gdm'
diag = 'GDM'
outcome = 'GDM'

gdm =  demographics[demographics['GDM'] == 1]


## PE ##
diag = 'pe_new'
outcome = 'pe_new'
pe =  demographics[demographics['pe_new'] == 1]

###PRETERM#######
diag = 'preterm_3c'
outcome = 'preterm_3c'

preterm =  demographics[demographics['preterm_3c'] != '3. Full term'].copy()
preterm.loc[:, outcome] = 1

df_list = [ghtn, gdm, preterm, pe]
'''

In [4]:
'''
vitals1 = pd.read_csv('vitals1.csv')
vitals2 = pd.read_csv('vitals2.csv')
vitals = pd.concat([vitals1, vitals2])

labs1 = pd.read_csv('labs1.csv')
labs2 = pd.read_csv('labs2.csv')
labs = pd.concat([labs1, labs2])
'''

In [5]:
'''
### Lab preprocessing ### 

#making months before last menstural period negative time
labs['mos_priorlmp'] = -1.0 * labs['mos_priorlmp']

#making one time column instead of prior/after
labs['time'] = labs['mos_afterlmp'].fillna(labs['mos_priorlmp'])

#Converting to weeks
labs['time'] = labs['time']*30.4375/7

#only taking labs two years before and up to cutoff
labs['time'] = np.round(labs['time']).astype(int)  
labs = labs[(labs['time'] >= -104) & (labs['time'] <= 36)]

labs = labs.drop(['mos_priorlmp', 'mos_afterlmp'], axis=1)
'''

In [6]:
'''
labs = labs.set_index('PREG_ID')
'''

In [7]:
'''
vitals['mos_priorlmp'] = -1.0 * vitals['mos_priorlmp']
vitals['time'] = vitals['mos_afterlmp'].fillna(vitals['mos_priorlmp'])

vitals['time'] = vitals['time']*30.4375/7

vitals['time'] = np.round(vitals['time']).astype(int) 
vitals = vitals[(vitals['time'] >= -104) & (vitals['time'] <= 36)]
vitals = vitals.drop(['mos_priorlmp', 'mos_afterlmp'], axis=1)

#Separate out HR, systolic, and diastolic into separate rows. 
vitals = pd.melt(vitals, id_vars=['preg_id', 'time'], var_name='TEST_TYPE', value_name='KPNC_RESULTN')
'''

In [8]:
'''
vitals.rename(columns={'preg_id': 'PREG_ID'}, inplace=True)
vitals = vitals.set_index('PREG_ID')
'''

In [ ]:
'''
labs2 = labs[['time', 'TEST_TYPE', 'KPNC_RESULTN']].copy()

tot = pd.concat([vitals, labs2])
print(tot.head())
'''

In [ ]:
'''
print(labs.shape)
print(vitals.shape)
print(tot.shape)
print(tot.index.nunique())
'''

In [ ]:
'''
his = pd.read_csv('history.csv')
for i in range(len(df_list)):
    df = df_list[i]
    df.rename(columns={'preg_id': 'PREG_ID'}, inplace=True)
    df = df.set_index('PREG_ID') 
    df = pd.merge(df, his, how = 'left', left_index = True, right_index = True)
    df = df.loc[:, ['mrace', 'parity_c', 'mage']]
    df['parity'] = (demographics['parity_c'] == '1. 0') * 1
    df = tot.merge(df, how='inner', left_index=True, right_index=True)
    df_list[i] = df
''' 

In [16]:
'''
feats = ['sbp_avg', 'gttpm_1_max', 'sbp_avg', 'dbp_max']
print(tot['TEST_TYPE'].unique().tolist())
'''

In [ ]:
'''
cols = tot['TEST_TYPE'].unique().tolist()

for df in df_list:
    df = df.dropna(subset=['KPNC_RESULTN'])

    
nghtn = df_list[0].index.nunique()
ngdm = df_list[1].index.nunique()
npreterm = df_list[2].index.nunique()
npe = df_list[3].index.nunique()

ghtn = df_list[0][df_list[0]['TEST_TYPE'] == 'avgbp_systolic']
gdm = df_list[1][df_list[1]['TEST_TYPE'] == 'GTTPM_1']
preterm = df_list[2][df_list[2]['TEST_TYPE'] == 'avgbp_diastolic']
pe = df_list[3][df_list[3]['TEST_TYPE'] == 'avgbp_systolic']

#change time range based on graph you want
time_range = range(-104, 0)

seen_ghtn = set()
seen_gdm = set()
seen_preterm = set()
seen_pe = set()

cumulative_ghtn = []
cumulative_gdm = []
cumulative_preterm = []
cumulative_pe = []

for t in time_range:
    current_ghtn = set(ghtn[ghtn['time'] == t].index)
    current_gdm = set(gdm[gdm['time'] == t].index)
    current_preterm = set(preterm[preterm['time'] == t].index)
    current_pe = set(pe[pe['time'] == t].index)
    
    seen_ghtn.update(current_ghtn)
    seen_gdm.update(current_gdm)
    seen_preterm.update(current_preterm)
    seen_pe.update(current_pe)

    cumulative_ghtn.append(len(seen_ghtn))
    cumulative_gdm.append(len(seen_gdm))
    cumulative_preterm.append(len(seen_preterm))
    cumulative_pe.append(len(seen_pe))

fin_ghtn = (np.array(cumulative_ghtn) / nghtn) * 100    
fin_gdm = (np.array(cumulative_gdm) / ngdm) * 100    
fin_preterm = (np.array(cumulative_preterm) / npreterm) * 100    
fin_pe = (np.array(cumulative_pe) / npe) * 100  

print(fin_ghtn[-1])
print(fin_gdm[-1])
print(fin_preterm[-1])
print(fin_pe[-1])
plt.figure(figsize=(10, 10))

#plt.axhline(y=95.5998, color='orange', linestyle='--', alpha=0.5, linewidth=3)
#plt.axhline(y=56.4849, color='green', linestyle='--', alpha=0.5, linewidth=3)
#plt.axhline(y=95.747, color='teal', linestyle='--', alpha=0.5, linewidth=3)
#plt.axhline(y=95.35, color='red', linestyle='--', alpha=0.5, linewidth=3)


#plt.axhline(y=95.5998, color='orange', linestyle='--', alpha=0.5, linewidth=3)
#plt.axhline(y=56.484, color='green', linestyle='--', alpha=0.5, linewidth=3)
#plt.axhline(y=93.787, color='teal', linestyle='--', alpha=0.5, linewidth=3)
#plt.axhline(y=93.6022, color='red', linestyle='--', alpha=0.5, linewidth=3)


plt.plot(time_range, fin_ghtn, label='GHTN: SBP', color='orange', linewidth=3)
plt.plot(time_range, fin_gdm, label='GDM: OGTT-1h', color='green', linewidth=3)
plt.plot(time_range, fin_pe, label='PE: DBP', color='teal', linewidth=3)
plt.plot(time_range, fin_preterm, label='Preterm: SBP', color='red', linewidth=3)

#plt.xlabel('Time (weeks)', fontsize=24)
plt.ylabel('Percent of Pregnancies', fontsize=24)
plt.title('Pre-Pregnancy', fontsize=24)
plt.tick_params(axis='both', which='major', labelsize=24)
plt.ylim(-1,100)
plt.legend(fontsize=20)
plt.grid(True)
plt.show()
'''